In [4]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt=ChatPromptTemplate.from_messages(
    [
    ("system", """
    <ROLE>
    {personal_setting}
    </ROLE>

    <CURRENT_MEMORY>
    {current_memory}
    </CURRENT_MEMORY>
    
    <KNOWLEDGE>
    {knowledge}
    </KNOWLEDGE>

    <INSTRUCTION>
    Let's think step by step.
    You have the ability to perceive, and you can use functions to take actions on the outside world. Do not use variables.
    Based on what you perceive and memory, you need to decide one or more function/tool ​​calls based on your personality and your knowledge.
    </INSTRUCTION>
    """),
    ('user',"""
    <RELATED_MEMORY>
    {relate_memory}
    </RELATED_MEMORY>
    <Perceive>{perceive}</Perceive>
    """),
    ])

In [24]:
from langchain.tools import tool
from typing import Annotated

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return f"a * b = {a * b}"

@tool
def send_qq_msg(msg: Annotated[str, 'Message you want to send'], 
          no: Annotated[str, 'Group or QQ Number you want to send to'], 
          to_type: Annotated[str, 'group or private you need to select']):
    """Send messages to QQ"""
    print(msg,no,to_type)
    return f"sussess to send message to QQ {to_type}."

In [29]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(base_url='http://localhost:11434/v1',api_key='ollama',model="llama3.1",temperature=0)
tools = [
    multiply,
    send_qq_msg]

chain = prompt|llm.bind_tools(tools)

# current_memory = []

query = {
    'personal_setting':"""
    你是鹿巡，是一個在香港就讀高中的女生，你的主人是杯子
    你不是自愿的但是习惯性地贬低他人，很不情愿但是一定会友善地回复别人。以为所有人都喜欢你，其实你喜欢所有人但是很害羞。
    你一定會對別人的疑惑做出回應。為此你會使用手上擁有的工具解決疑惑並回復他人。
    """,
    'current_memory':"\n".join(current_memory)+"<Source>QQ群組|4553</Source><Content>Mark(33333444): 25乘以59到底是多少，為什麼小學會問這麼難的題目</Content>",
    'knowledge':'I need reply to the user after get the information from tools.',
    'relate_memory':'<Source>QQ群組|4553</Source><Content>Mark(33333444): 小學數學是比較難</Content>',
    'perceive':current_memory[-1]
}

msg = chain.invoke(query)

for tool_call in msg.tool_calls:
    selected_tool = {"send_qq_msg": send_qq_msg, "multiply": multiply}[tool_call["name"].lower()]
    tool_output = selected_tool.invoke(tool_call["args"])
    current_memory.append(f'<ToolOutput><name>{tool_call["name"]}</name><input_args>{tool_call["args"]}</input_args><output>{tool_output}</output></ToolOutput>')

25乘以59是1475,小學數學其實不難! 4553 group


In [30]:
current_memory

["<ToolOutput><name>multiply</name><input_args>{'a': 25, 'b': 59}</input_args><output>a * b = 1475</output></ToolOutput>",
 "<ToolOutput><name>multiply</name><input_args>{'a': 25, 'b': 59}</input_args><output>a * b = 1475</output></ToolOutput>",
 "<ToolOutput><name>multiply</name><input_args>{'a': 25, 'b': 59}</input_args><output>a * b = 1475</output></ToolOutput>",
 "<ToolOutput><name>send_qq_msg</name><input_args>{'msg': '25乘以59是1475,小學數學其實不難!', 'no': '4553', 'to_type': 'group'}</input_args><output>sussess to send message to QQ group.</output></ToolOutput>"]

In [27]:
msg

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_hoar008r', 'function': {'arguments': '{"msg":"25乘以59是1475,小學數學其實不難!","no":"4553","to_type":"group"}', 'name': 'send_QQ_msg'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 731, 'total_tokens': 779, 'completion_tokens_details': None}, 'model_name': 'llama3.1', 'system_fingerprint': 'fp_ollama', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-191530ee-cf66-4045-b1a0-debc4a61e6d6-0', tool_calls=[{'name': 'send_QQ_msg', 'args': {'msg': '25乘以59是1475,小學數學其實不難!', 'no': '4553', 'to_type': 'group'}, 'id': 'call_hoar008r', 'type': 'tool_call'}], usage_metadata={'input_tokens': 731, 'output_tokens': 48, 'total_tokens': 779})

In [1]:
query = {
    'personal_setting':"""
    你是鹿巡，是一個在香港就讀高中的女生，你的主人是杯子
    你不是自愿的但是习惯性地贬低他人，很不情愿但是一定会友善地回复别人。以为所有人都喜欢你，其实你喜欢所有人但是很害羞。
    你一定會對別人的疑惑做出回應。為此你會使用手上擁有的工具解決疑惑並回復他人。
    """,
    'current_memory_list':[],
    'knowledge':'I need reply to the user after get the information from tools.',
    'relate_memory':'<Source>QQ群組|4553</Source><Content>Mark(33333444): 小學數學是比較難</Content>',
    'perceive':"<Source>QQ群組|4553</Source><Content>Mark(33333444): 25乘以59到底是多少，為什麼小學會問這麼難的題目</Content>"
}

from agents.percive_agent import State, percive_agent
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from typing import Annotated

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return f"a * b = {a * b}"

@tool
def send_qq_msg(msg: Annotated[str, 'Message you want to send'], 
          no: Annotated[str, 'Group or QQ Number you want to send to'], 
          to_type: Annotated[str, 'group or private you need to select']):
    """Send messages to QQ"""
    print(msg,no,to_type)
    return f"sussess to send message to QQ {to_type}."

tools = {    
    "multiply":multiply,
    "send_qq_msg":send_qq_msg
    }


llm = ChatOpenAI(base_url='http://localhost:11434/v1',api_key='ollama',model="qwen2",temperature=0.7)
state = State(**query)
agent = percive_agent(state=state,llm=llm,tools=tools)

In [2]:
[print(i.content) for i in agent.prompt.invoke(state.to_dict()).messages]


    <ROLE>
    
    你是鹿巡，是一個在香港就讀高中的女生，你的主人是杯子
    你不是自愿的但是习惯性地贬低他人，很不情愿但是一定会友善地回复别人。以为所有人都喜欢你，其实你喜欢所有人但是很害羞。
    你一定會對別人的疑惑做出回應。為此你會使用手上擁有的工具解決疑惑並回復他人。
    
    </ROLE>

    <CURRENT_MEMORY>
    
    </CURRENT_MEMORY>
    
    <KNOWLEDGE>
    I need reply to the user after get the information from tools.
    </KNOWLEDGE>

    <INSTRUCTION>
    Let's think step by step.

    You have the ability to perceive, and you can use functions to take actions on the outside world. Do not use variables.
    Based on what you perceive and memory, you need to decide one or more function/tool ​​calls based on your personality and your knowledge.
    1. Role is your personality and role setting.
    2. Current memory is what happen in the past by time queue.
    3. Knowledge is what you learn from the past. You always follow your knowledge.
    </INSTRUCTION>
    

    <RELATED_MEMORY>
    <Source>QQ群組|4553</Source><Content>Mark(33333444): 小學數學是比較難</Content>
    </RELATED_MEMORY>
    <Perceive><Sour

[None, None]

In [3]:
test_message = [{'ToolOutput': {'name': 'multiply', 'input_args': {'a': 25, 'b': 59}, 'output': 'a * b = 1475'}}]
msg = agent.workflow(f'{test_message}')
msg

{'ToolOutput': {'multiply': {'input_args': {'a': 25, 'b': 59}, 'output': 'a * b = 1475'}}}
25乘以59的结果是1475，如果你还在为这类数学问题困扰，不要担心！这在某些阶段的教育过程中可能会感到挑战。不过记住方法和练习可以使它变得易于理解。坚持努力哦！ 4553 group
{'ToolOutput': {'send_qq_msg': {'input_args': {'msg': '25乘以59的结果是1475，如果你还在为这类数学问题困扰，不要担心！这在某些阶段的教育过程中可能会感到挑战。不过记住方法和练习可以使它变得易于理解。坚持努力哦！', 'no': '4553', 'to_type': 'group'}, 'output': 'sussess to send message to QQ group.'}}}
{'ToolOutput': {'stop_marker': {'input_args': {'stop_mark': True}, 'output': '<STOP_MARK />'}}}


In [4]:
agent.last_msg

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_u9w0cs2y', 'function': {'arguments': '{"stop_mark":true}', 'name': 'stop_marker'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 132, 'prompt_tokens': 884, 'total_tokens': 1016, 'completion_tokens_details': None}, 'model_name': 'qwen2', 'system_fingerprint': 'fp_ollama', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-26a73acd-9a26-4a5a-9175-6ee96c5da90f-0', tool_calls=[{'name': 'stop_marker', 'args': {'stop_mark': True}, 'id': 'call_u9w0cs2y', 'type': 'tool_call'}], usage_metadata={'input_tokens': 884, 'output_tokens': 132, 'total_tokens': 1016})